# Lab 04: Data Formats: XML and JSON

*INFO 4617/5617 · Week 4 · Chapter 4, [Data Formats: XML and JSON](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-04-data-formats.html)*

Today we code together. Each step below says what we're building and why, and its code cell holds only the comments that frame it: we write the code live, in class. Nothing here is turned in. After each step, a **💬 Discuss** box holds questions to talk over before the next one; click it to open it. The **⏱ Microsprints** are short tasks for you alone or with a neighbor, each ending in something to share with the room.

We'll parse one real source in each format. **XML:** the roster of the U.S. House of Representatives that the Office of the Clerk publishes, the chapter's case. We'll walk its tree, count its parties, read what the count hides, and turn it into a DataFrame. **JSON:** a year of daily weather for Boulder from NASA's POWER API, which we'll take apart one key at a time, turn into a DataFrame, and plot. The chapter's JSON case, the Open-Meteo forecast, is yours in the Recommended Exercises.

**Before class:** open this notebook in the `webdata` environment (chapter 1), and have your email address ready for the User-Agent. Nothing today needs a key or an account.

On Friday, run every cell above where we stopped (**Run → Run All Above Selected Cell**).

## Step 0 — Set up (4 minutes)

Import what today needs: `requests` to fetch, [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) to parse XML, Python's built-in [`json`](https://docs.python.org/3/library/json.html) module for JSON strings, `Counter` to tally, `time` to pause between requests, pandas, and matplotlib. Then define `HEADERS` with the book's User-Agent, as in week 2, and send it with every request. BeautifulSoup's `"xml"` parser comes from the `lxml` package, which the `webdata` environment includes; if Step 2 stops with `bs4.FeatureNotFound`, the chapter's callout in "Parsing with BeautifulSoup" says how to add it.

In [ ]:
# Import requests, json, time, Counter (from collections), BeautifulSoup (from bs4), pandas as pd, matplotlib.pyplot as plt
# Define HEADERS: a dict whose "User-Agent" is "WebDataScience/1.0 (your-email@colorado.edu)", with your email
# Your code here

<details>
<summary>💬 Discuss</summary>

- In week 2 we wrote this User-Agent. Who reads it, and what can they do with the email address in it?
- Neither site we use today refuses a request without one. Why send it anyway?

</details>

## Step 1 — Fetch the roster, and look before you parse (6 minutes)

The Clerk publishes the roster at `https://clerk.house.gov/xml/lists/MemberData.xml`. Open that address in Chrome first: a file with no stylesheet is drawn as a tree you can fold, as in the chapter's figure of it. Then fetch it with `requests.get()`, as the chapter does. Before parsing anything, look at three things: the status code, the `Content-Type` header, and the first bytes of [`response.content`](https://requests.readthedocs.io/en/latest/api/#requests.Response.content), the body as the server sent it. The header names no character set, so `requests` guesses one, `response.encoding`: compare its guess with the encoding the file declares in its first line. The chapter's "Working with Real XML" explains why this matters.

In [ ]:
# Define ROSTER, the roster's address, and fetch it with HEADERS
# Print the status code, the Content-Type header, the size in bytes, and the encoding requests guessed
# Print the first 150 bytes of response.content: which encoding does the file itself declare?
# Your code here

<details>
<summary>💬 Discuss</summary>

- `clerk.house.gov/robots.txt` answers 404. What did week 2 say a missing `robots.txt` means, and what doesn't it mean?
- Our code assumes the Clerk answers. If it sent a 503 and an HTML error page instead, which line today would stop, or would we just find zero members? What would you print first to tell?
- Your Recommended Exercises start the same way with an RSS feed. What would you check here before writing any parsing code?

</details>

## Step 2 — Walk the tree: `find`, `find_all`, text, and attributes (10 minutes)

`BeautifulSoup(response.content, "xml")` turns the bytes into a tree; hand it the bytes, not `response.text`, so that it reads the encoding the file declares. There are two ways down the tree. Dot notation (`soup.members.member`) takes the first tag of that name, and works only for names that are Python identifiers. [`.find()`](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#find) and [`.find_all()`](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#find-all) search the whole subtree, so they reach `<firstname>` inside `<member-info>` and work for hyphenated names like `<state-fullname>`. A tag's `.contents` lists its direct children. Text sits between tags and is read with `.text`; attributes sit inside the opening tag and are read with brackets, like a dictionary. The root's `publish-date` attribute says which snapshot of the House this is. See the chapter's "Parsing with BeautifulSoup".

In [ ]:
# Parse response.content with the "xml" parser; print the root MemberData tag's publish-date attribute and its children's names
# Take the first member with dot notation and print its statedistrict; with .find(), its lastname and party (text),
#   its state's postal-code (an attribute), and its state-fullname
# find_all the member tags into members, and print how many there are
# Your code here

### ⏱ Microsprint (5 minutes, alone or in pairs)

**Do:** the chapter says that parsing `response.text` would garble accented names. Check it: parse `response.text` into a second tree, and compare its last names with ours, member by member.
**Share:** how many last names differ between the two trees, and one as each tree spells it.

In [ ]:
# Microsprint: parse response.text into a second tree, garbled; make a list of last names from each tree
# Pair the two lists with zip(), and print each pair that differs
# Your code here

<details>
<summary>💬 Discuss</summary>

- Why does `soup.member-info` fail? What does Python think you wrote?
- The second tree parsed without an error or a warning. How would you have noticed the garbled names, and in which week 1 attribute of the response was the clue?
- The roster keeps the state's postal code in an attribute and its full name in a child tag. If you designed the file, which would you choose, and why?

</details>

## Step 3 — Count, then read the count (8 minutes)

[`Counter`](https://docs.python.org/3/library/collections.html#collections.Counter) tallies a list. Collect every member's `<party>` text and count them. Then read the odd entries instead of discarding them. An empty party is a vacant seat: its `<member>` block is still there with every tag empty, and a `<footnote>` says why. And the schema records two things about each member, `<party>` (formal registration) and `<caucus>` (the party they sit with), which don't always agree. See the end of the chapter's "Working with Real XML".

In [ ]:
# Count the members' party text with Counter
# For each member whose party is empty, print its statedistrict and its footnote
# Count (party, caucus) pairs with Counter: which pair doesn't match?
# Your code here

<details>
<summary>💬 Discuss</summary>

- The roster has 441 entries. How many representatives are serving, and what would a paper that reported 441 have gotten wrong?
- One member's party is `I` and his caucus is `R`. Name a research question for which he counts as a Republican, and one for which he doesn't.
- Week 3 said to record the date with every collection, because one run is a snapshot. What does `publish-date` let a reader of your analysis do that the counts alone don't?

</details>

## Step 4 — From XML to a DataFrame (10 minutes)

The chapter's pipeline: `find_all()` the records, pull the fields you need out of each one into a dictionary, collect the dictionaries in a list, and pass the list to [`pd.DataFrame()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html). The state comes from the `postal-code` attribute, not from text. Then count the rows per state with `groupby()`. This is the chapter's code from "From XML to DataFrame", with one change: the chapter guards each field with `if m.find(...) else None`, in case a tag is missing, and we leave the guards out to keep the loop short. The Discuss box asks what that costs.

In [ ]:
# For each member, build a dict record: first_name, last_name, party, state (the state tag's postal-code attribute),
#   and district; collect the records in a list and make the DataFrame df
# Print df.head() and the 10 states with the most rows
# Your code here

### ⏱ Microsprint (5 minutes, alone or in pairs)

**Do:** drop the vacant seats, then make a table of state by party with [`pd.crosstab()`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html). Count the states whose delegation has at least one Democrat and one Republican, and look up Colorado's row.
**Share:** the number of split states, and Colorado's split.

In [ ]:
# Microsprint: keep the rows whose party isn't empty, then pd.crosstab() state by party
# Count the states with at least one D and one R, and print Colorado's row
# Your code here

<details>
<summary>💬 Discuss</summary>

- We left out the chapter's `if m.find(...) else None`. If one member had no `<district>` tag, what would our loop do, and what would the chapter's do?
- The vacancies have every tag, just empty. Would the chapter's guards have caught them? What did?
- In week 1, one call, `pd.DataFrame(data["items"])`, turned the pageviews JSON into a DataFrame. Why did the roster need a loop?

</details>

## Step 5 — JSON is lists and dictionaries (9 minutes)

A JSON array is a Python list, and a JSON object is a Python dictionary. [`json.dumps()`](https://docs.python.org/3/library/json.html#json.dumps) writes Python as a JSON string, and [`json.loads()`](https://docs.python.org/3/library/json.html#json.loads) parses one back (the `s` is for *string*). Take the roster's first two rows as a list of dictionaries with [`to_dict("records")`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_dict.html), write them as JSON, parse the string back, and make a DataFrame again: the list-of-dictionaries pattern runs both ways. See the chapter's "The List-of-Dictionaries Pattern" and "Parsing JSON Strings".

In [ ]:
# Take df's first two rows as a list of dicts (to_dict("records")) and print them as JSON (json.dumps, indent=2)
# Parse that string back with json.loads; print its type and the first record's last_name, then make it a DataFrame
# Your code here

### ⏱ Microsprint (4 minutes, alone or in pairs)

**Do:** `json.loads()` three broken strings, the kinds the chapter's "Common Issues to Debug" lists: single quotes, a trailing comma, and an HTML error page where JSON should be. Errors are this microsprint's subject, so it catches them with `try`/`except` to see all three messages in one run.
**Share:** each error message, and which of the three you'd expect from a live API.

In [ ]:
# Microsprint: json.loads() each of "{'name': 'Colorado'}", '{"name": "Colorado",}', and "<html><body>Too Many Requests</body></html>",
#   catching json.JSONDecodeError and printing its message
# Your code here

<details>
<summary>💬 Discuss</summary>

- Every value in our records is a string, because XML gave us text. In week 1's pageviews JSON, what type was `views`, and who decided?
- If `.json()` fails on a live response, the body is probably an error page. Which two things from week 1's tour of the response would you print first to find out?
- The House publishes its roster as XML, and NASA its weather as JSON. Using the chapter's "When to Use Which", why might each have chosen its format?

</details>

## Step 6 — Peel the onion: a year of Boulder's weather (9 minutes)

NASA's [POWER API](https://power.larc.nasa.gov/docs/services/api/temporal/daily/) serves daily weather for any point on Earth, with no key, from MERRA-2, a model of the atmosphere fitted to observations. Its address is `https://power.larc.nasa.gov/api/temporal/daily/point`. Ask for the chapter's Boulder coordinates and the whole of 2025: daily high (`T2M_MAX`), low (`T2M_MIN`), and precipitation (`PRECTOTCORR`). The parameters go in a dictionary, as in the chapter, and `print(response.url)` shows the full address `requests` built, which you can paste into Chrome with **Pretty-print** ticked. Then peel the onion, as the chapter's "Handling Deeply Nested JSON" does: `type()` and `.keys()` at the top, then one key deeper at a time, until you reach the values. Use `.get()` for a key that might be missing. A past year gives the same answer every time you ask; a forecast never does.

In [ ]:
# Define POWER, the API's address, and params: parameters "T2M_MAX,T2M_MIN,PRECTOTCORR", community "AG", format "JSON",
#   latitude 40.01, longitude -105.27, start "20250101", end "20251231"
# Fetch with params and HEADERS, parse with .json() into data, and print response.url, type(data), and data.keys()
# Your code here

In [ ]:
# One key deeper at a time: print data["geometry"]["coordinates"], the keys under "properties" and then under
#   "parameter", data["parameters"] (each variable's units), and data.get("messages")
# Your code here

<details>
<summary>💬 Discuss</summary>

- The coordinates came back as `[-105.27, 40.01, 1801.15]`. Which number is the latitude, how would you know, and what is the third?
- POWER's `robots.txt` disallows `/api/` for crawlers, as Wikipedia's disallows `/w/` (week 2). Are we a crawler?
- Our code assumes POWER answers. If it sent a 429 and an HTML page instead, which line would stop first, with which of Step 5's messages? Does the chapter's Open-Meteo code guard against that?

</details>

## Step 7 — Into a DataFrame, and a plot (12 minutes)

`data["properties"]["parameter"]` is a dictionary of dictionaries: one per variable, each keyed by date. `pd.DataFrame()` lines them up by those keys, one column per variable and one row per date. Turn the index into dates with [`pd.to_datetime()`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) and `format="%Y%m%d"`, and replace POWER's fill value for a missing day, which `data["header"]` names, with `NaN`. Then plot the highs and lows as the chapter plots its forecast, with [`fill_between()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.fill_between.html) between them. We start the plot with [`plt.subplots()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html), the course's one way to plot; the chapter's `plt.figure()` draws the same thing. The chapter's forecast holds *parallel lists* instead, matched only by position; these values are matched by date.

In [ ]:
# Make weather, a DataFrame of data["properties"]["parameter"]; turn its index into dates (format="%Y%m%d"),
#   replace data["header"]["fill_value"] with float("nan"), and rename the columns high, low, and precip
# Print the first rows, the number of days, and the number of missing values in each column
# Your code here

In [ ]:
# With fig, ax = plt.subplots(), plot high and low with fill_between between them; add a title, axis labels, and a legend
# Your code here

### ⏱ Microsprint (5 minutes, alone or in pairs)

**Do:** a short version of Additional Exercise 3, on our year instead of a forecast. Count the nights whose low fell below 0 °C, and find the last one before July and the first one after. If there's time, redraw the lows with a line at 0 °C ([`ax.axhline()`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.axhline.html)) and the freezing nights marked.
**Share:** the number of freezing nights, and the dates of the last spring frost and the first autumn frost.

In [ ]:
# Microsprint: keep the days whose low was below 0 °C in frost; print how many, the last before July, and the first after June
# Your code here

In [ ]:
# Microsprint, if there's time: plot the lows, a horizontal line at 0 °C, and the freezing nights as dots, with a title and axis labels
# Your code here

<details>
<summary>💬 Discuss</summary>

- If one day were missing from `T2M_MIN`, what would `pd.DataFrame()` do with it? What if a day were missing from one of the forecast's parallel lists?
- The model's grid cell sits about 150 m above the town, and spans tens of kilometers. Would you write "Boulder had this many freezing nights in 2025"? What would you write instead?

</details>

## Step 8 — One function, more places (12 minutes)

Now we want the same table for more than one place, so this is where a function earns its place. Gather the lines we wrote in Steps 6 and 7 into `get_daily(latitude, longitude, year)`, which returns the DataFrame and the grid cell's elevation; nothing in it is new. Then loop over Boulder and Leadville, at about 3,100 m the highest incorporated city in the United States, and turn each place into one row. The loop pauses a second after each request with `time.sleep(1)`, as week 2's chapter does. Boulder's row should match Step 7's numbers: that's our check that the function does what the plain lines did.

In [ ]:
# Define get_daily(latitude, longitude, year): Step 6's params for that point and year, the request and .json(),
#   then Step 7's DataFrame; return the DataFrame and the elevation (the third coordinate)
# Your code here

In [ ]:
# For each of places, get_daily() for 2025 and append a dict to rows: place, grid elevation, mean high, and nights below 0 °C
# Sleep a second after each request; then print rows as a DataFrame
# Your code here

### ⏱ Microsprint (5 minutes, alone or in pairs)

**Do:** call `get_daily()` for a place you know, such as your hometown (its Wikipedia article gives its coordinates; west and south are negative). The key uses Grand Junction, at 39.06, -108.55.
**Share:** your place, its grid elevation, its mean high, and its freezing nights in 2025, beside Boulder's row.

In [ ]:
# Microsprint: get_daily() for your place in 2025; print its grid elevation, mean high, and nights below 0 °C
# Your code here

<details>
<summary>💬 Discuss</summary>

- Leadville's grid cell is about 1,400 m higher than Boulder's. Is the gap in mean highs about what you'd expect, and what else besides height could explain it?
- If each of us looped `get_daily()` over 50 places, how many requests would this room send, and how long would the `sleep` make each loop take (week 2)?
- We wrote `get_daily()` only once we needed it twice. What would you have to change to run the same comparison on the chapter's Open-Meteo forecast, and why couldn't you compare it with these numbers?

</details>

## Step 9 — What did we learn? (5 minutes)

In a few sentences, as comments or in a Markdown cell:

- Of the roster's entries, how many are people serving today? What in the XML told you about the rest, and what would `Counter(parties)` alone have told you?
- How many last names would `response.text` have garbled, and would anything have warned you?
- How many freezing nights did Boulder's grid cell have in 2025, and how do Leadville and your place compare? What does the grid elevation say about what these numbers describe?
- Besides the data, what did each source tell you about itself (`publish-date`, units, `fill_value`)? Which of those would you save beside your DataFrame?

**Recommended Exercises ahead** (in the chapter's companion notebook, due Sunday):

- Step 1, `HEADERS` and an RSS feed you've checked against its `robots.txt`: Step 0's `HEADERS`, and Step 1's question about the Clerk's missing `robots.txt`. [`handouts/week-04/rss-feeds.md`](../../handouts/week-04/rss-feeds.md) lists starter feeds.
- Step 2, fetch the feed and look at the raw XML: Step 1's status code, `Content-Type`, and first bytes.
- Step 3, parse it and count the items: Step 2's `"xml"` parser and `find_all()`.
- Step 4, the items into a DataFrame: Step 4's list-of-dictionaries loop. In a feed the repeating tag is `<item>`.
- Step 5, fetch the forecast and map its keys: Step 6's peeling, one key at a time.
- Step 6, a DataFrame from the forecast's parallel lists, and a plot: Step 7's DataFrame and plot. Parallel lists line up by position, not by date: the chapter's "Handling Deeply Nested JSON" shows how to build a DataFrame from them.
- Step 7, interpret: Steps 3 and 7's Discuss boxes.

In [ ]:
# Your answers here, as comments


## Friday: from the lab to the textbook (15 minutes)

What did the lab turn up that the chapter could say better? Some places to look:

- Step 2 printed the roster's top-level tags. The chapter's "Working with Real XML" names two blocks we ignore, `<title-info>` and each member's `<committee-assignments>`. Is that the whole list, and does the third block hold any tag names that `soup.find()` could pick up by mistake?
- The chapter's printed output says which date the roster was published. What did Step 2 print, and do the chapter's counts still hold?
- Step 4 left out the chapter's `if m.find(...) else None` guards, and every member still parsed, vacancies included. Should the chapter say which tags can be missing in this file, or show a case where a guard matters?
- The chapter fetches the roster and the forecast without `headers=`, though its Recommended Exercises (and week 2) ask for a User-Agent. Should its code send one?
- A microsprint that would make a good Additional Exercise, such as the `response.text` check, the broken JSON, or the freezing nights.
- A sentence in the chapter that confused you before today, and one you could now explain to someone else.

[`handouts/common/revision-framework.md`](../../handouts/common/revision-framework.md) says how to propose a revision.